<a href="https://colab.research.google.com/github/vivekcodes-tech/CineSearch/blob/main/Cinesearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import urllib.request
import zipfile

# Define the download URL and paths
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"
zip_destination = "ml-32m.zip"
extract_directory = "ml-32m-data"

# 1. Download the Dataset Zip File
if not os.path.exists(zip_destination):
    print("Downloading MovieLens 32M zip file... (this may take a couple of minutes)")
    urllib.request.urlretrieve(dataset_url, zip_destination)
    print("Download complete!")
else:
    print("MovieLens 32M zip file already exists in local workspace.")

# 2. Extract files
if not os.path.exists(extract_directory):
    print("Extracting zip archive...")
    with zipfile.ZipFile(zip_destination, 'r') as zip_ref:
        zip_ref.extractall(extract_directory)
    print("Extraction complete!")
else:
    print("Archive files are already extracted.")

# 3. Verify existence and show file sizes
expected_files = ['movies.csv', 'ratings.csv', 'tags.csv', 'links.csv']
target_folder = os.path.join(extract_directory, 'ml-32m')

print("\n--- Verification Results ---")
if os.path.exists(target_folder):
    for file_name in expected_files:
        full_path = os.path.join(target_folder, file_name)
        if os.path.exists(full_path):
            file_size_mb = os.path.getsize(full_path) / (1024 * 1024)
            print(f"File: {file_name:<12} | Size: {file_size_mb:>8.2f} MB | Status: OK")
        else:
            print(f"File: {file_name:<12} | Status: MISSING (Please check extraction)")
else:
    print(f"Error: The directory {target_folder} does not exist.")

Download complete!
Extracting zip archive...
Extraction complete!

--- Verification Results ---
File: movies.csv   | Size:     4.05 MB | Status: OK
File: ratings.csv  | Size:   836.45 MB | Status: OK
File: tags.csv     | Size:    69.00 MB | Status: OK
File: links.csv    | Size:     1.86 MB | Status: OK


In [ ]:
import os
import pandas as pd
import numpy as np

data_dir = 'ml-32m-data/ml-32m'

# Helper to print header
def print_section(title):
    print(f"\n{'='*40}\n{title}\n{'='*40}")

# 1. Analyze movies.csv
print_section("Analyzing movies.csv")
movies_df = pd.read_csv(os.path.join(data_dir, 'movies.csv'))
print(f"Row count: {len(movies_df):,}")
print("Columns and Null values:")
print(movies_df.isnull().sum())
print("\nFirst 3 rows:")
display(movies_df.head(3))

# 2. Analyze tags.csv
print_section("Analyzing tags.csv")
tags_df = pd.read_csv(os.path.join(data_dir, 'tags.csv'))
print(f"Row count: {len(tags_df):,}")
print("Columns and Null values:")
print(tags_df.isnull().sum())
print(f"Unique tag texts: {tags_df['tag'].nunique():,}")

# 3. Analyze links.csv
print_section("Analyzing links.csv")
links_df = pd.read_csv(os.path.join(data_dir, 'links.csv'))
print(f"Row count: {len(links_df):,}")
print("Columns and Null values:")
print(links_df.isnull().sum())


Analyzing movies.csv
Row count: 87,585
Columns and Null values:
movieId    0
title      0
genres     0
dtype: int64

First 3 rows:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance



Analyzing tags.csv
Row count: 2,000,072
Columns and Null values:
userId        0
movieId       0
tag          17
timestamp     0
dtype: int64
Unique tag texts: 140,979

Analyzing links.csv
Row count: 87,585
Columns and Null values:
movieId      0
imdbId       0
tmdbId     124
dtype: int64


In [10]:
# 4. Memory-Efficient Analysis of ratings.csv via Chunking
import os
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import ssl

data_dir = 'ml-32m-data/ml-32m'
ratings_path = os.path.join(data_dir, 'ratings.csv')

# Helper to print section headers
def print_section(title):
    print(f"\n{'='*40}\n{title}\n{'='*40}")

# Automatically download and extract if the dataset is missing
if not os.path.exists(ratings_path):
    print_section("Downloading and Extracting Dataset")
    dataset_url = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"
    zip_destination = "ml-32m.zip"
    extract_directory = "ml-32m-data"

    if not os.path.exists(zip_destination):
        print("Downloading MovieLens 32M zip file... (this may take a couple of minutes)")
        # Create an unverified SSL context to bypass verification errors
        context = ssl._create_unverified_context()
        with urllib.request.urlopen(dataset_url, context=context) as response, open(zip_destination, 'wb') as out_file:
            out_file.write(response.read())
        print("Download complete!")

    print("Extracting zip archive...")
    with zipfile.ZipFile(zip_destination, 'r') as zip_ref:
        zip_ref.extractall(extract_directory)
    print("Extraction complete!")

print_section("Analyzing ratings.csv (Chunked Processing)")

chunksize = 2_000_000

total_rows = 0
unique_users = set()
unique_movies = set()

rating_sum = 0.0
rating_sq_sum = 0.0
rating_counts = {}

print("Processing ratings.csv in chunks...")
for chunk in pd.read_csv(ratings_path, chunksize=chunksize):
    total_rows += len(chunk)

    # Update unique sets
    unique_users.update(chunk['userId'].unique())
    unique_movies.update(chunk['movieId'].unique())

    # Update aggregate sums for mean and variance calculations
    rating_sum += chunk['rating'].sum()
    rating_sq_sum += (chunk['rating'] ** 2).sum()

    # Update value counts for rating distributions
    counts = chunk['rating'].value_counts()
    for rating_val, count in counts.items():
        rating_counts[rating_val] = rating_counts.get(rating_val, 0) + count

# Calculating Statistics
mean_rating = rating_sum / total_rows
# Variance formula: E[X^2] - (E[X])^2
variance_rating = (rating_sq_sum / total_rows) - (mean_rating ** 2)
std_dev_rating = np.sqrt(variance_rating)

print("Processing complete!\n")
print(f"Total Rating Rows (N): {total_rows:,}")
print(f"Unique Users: {len(unique_users):,}")
print(f"Unique Movies with Ratings: {len(unique_movies):,}")
print(f"Mean Rating (μ_r): {mean_rating:.4f}")
print(f"Rating Variance (σ_r²): {variance_rating:.4f}")
print(f"Rating Std Dev (σ_r): {std_dev_rating:.4f}")

print("\nRating Distribution:")
for r_val in sorted(rating_counts.keys()):
    print(f"  Rating {r_val:.1f}: {rating_counts[r_val]:,} times")


Download complete!
Extracting zip archive...
Extraction complete!

Analyzing ratings.csv (Chunked Processing)
Processing ratings.csv in chunks...
Processing complete!

Total Rating Rows (N): 32,000,204
Unique Users: 200,948
Unique Movies with Ratings: 84,432
Mean Rating (μ_r): 3.5404
Rating Variance (σ_r²): 1.1215
Rating Std Dev (σ_r): 1.0590

Rating Distribution:
  Rating 0.5: 525,132 times
  Rating 1.0: 946,675 times
  Rating 1.5: 531,063 times
  Rating 2.0: 2,028,622 times
  Rating 2.5: 1,685,386 times
  Rating 3.0: 6,054,990 times
  Rating 3.5: 4,290,105 times
  Rating 4.0: 8,367,654 times
  Rating 4.5: 2,974,000 times
  Rating 5.0: 4,596,577 times


In [11]:
import os
import pandas as pd

# Let's count the exact number of rating records in each subset dynamically to report our workloads
ratings_path = 'ml-32m-data/ml-32m/ratings.csv'
chunksize = 2_000_000

counts = {32: 0, 16: 0, 8: 0, 4: 0}

print("Counting rows for reproducible modulo stages (32, 16, 8, 4) in chunks...")
for chunk in pd.read_csv(ratings_path, chunksize=chunksize, usecols=['userId']):
    for mod in counts.keys():
        counts[mod] += (chunk['userId'] % mod == 0).sum()

print("\n--- Workload Subset Stats ---")
for mod, cnt in counts.items():
    print(f"Stage (userId % {mod:2} == 0): {cnt:,} rating rows")

Counting rows for reproducible modulo stages (32, 16, 8, 4) in chunks...

--- Workload Subset Stats ---
Stage (userId % 32 == 0): 991,618 rating rows
Stage (userId % 16 == 0): 1,954,519 rating rows
Stage (userId %  8 == 0): 3,878,599 rating rows
Stage (userId %  4 == 0): 7,862,045 rating rows


In [12]:
import sqlite3

db_path = 'cinesearch.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Enable foreign key enforcement inside SQLite runtime
cursor.execute("PRAGMA foreign_keys = ON;")

# 1. Create Movie Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS movie (
    movie_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    release_year INTEGER,
    imdb_id INTEGER,
    tmdb_id INTEGER
);
''')

# 2. Create Genre Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS genre (
    genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
    genre_name TEXT NOT NULL UNIQUE
);
''')

# 3. Create Movie Genre Junction Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS movie_genre (
    movie_id INTEGER NOT NULL,
    genre_id INTEGER NOT NULL,
    PRIMARY KEY (movie_id, genre_id),
    FOREIGN KEY (movie_id) REFERENCES movie(movie_id) ON DELETE CASCADE,
    FOREIGN KEY (genre_id) REFERENCES genre(genre_id) ON DELETE CASCADE
);
''')

# 4. Create App User Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS app_user (
    user_id INTEGER PRIMARY KEY
);
''')

# 5. Create Rating Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS rating (
    rating_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    movie_id INTEGER NOT NULL,
    rating REAL NOT NULL CHECK(rating >= 0.5 AND rating <= 5.0),
    timestamp INTEGER NOT NULL,
    FOREIGN KEY (user_id) REFERENCES app_user(user_id) ON DELETE CASCADE,
    FOREIGN KEY (movie_id) REFERENCES movie(movie_id) ON DELETE CASCADE,
    UNIQUE(user_id, movie_id)
);
''')

# 6. Create Tag Event Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS tag_event (
    tag_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    movie_id INTEGER NOT NULL,
    tag_text TEXT NOT NULL,
    timestamp INTEGER NOT NULL,
    FOREIGN KEY (user_id) REFERENCES app_user(user_id) ON DELETE CASCADE,
    FOREIGN KEY (movie_id) REFERENCES movie(movie_id) ON DELETE CASCADE
);
''')

conn.commit()
conn.close()
print("Database cinesearch.db initialized successfully with normalized tables and active key constraints!")

Database cinesearch.db initialized successfully with normalized tables and active key constraints!


In [13]:
import os
import re
import sqlite3
import pandas as pd

db_path = 'cinesearch.db'
data_dir = 'ml-32m-data/ml-32m'

# Helper function to extract release year
def extract_year(title):
    match = re.search(r'\((\d{4})\)\s*$', title)
    if match:
        return int(match.group(1)), re.sub(r'\s*\(\d{4}\)\s*$', '', title).strip()
    return None, title.strip()

# 1. Establish database connection
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

print("Processing and inserting genres...")
# 2. Extract and insert unique genres
movies_df = pd.read_csv(os.path.join(data_dir, 'movies.csv'))
all_genres = set()
for genres_str in movies_df['genres'].dropna():
    if genres_str != "(no genres listed)":
        all_genres.update(genres_str.split('|'))

genre_map = {}
cursor.execute("BEGIN TRANSACTION;")
for g in sorted(all_genres):
    cursor.execute("INSERT OR IGNORE INTO genre (genre_name) VALUES (?);", (g,))
cursor.execute("COMMIT;")

# Build map of genre_name to genre_id
cursor.execute("SELECT genre_id, genre_name FROM genre;")
for gid, gname in cursor.fetchall():
    genre_map[gname] = gid

print(f"Inserted {len(genre_map)} unique genres.")

# 3. Load Links for O(1) metadata lookup
print("Loading link metadata...")
links_df = pd.read_csv(os.path.join(data_dir, 'links.csv'))
# Convert to dictionary mapping movieId -> (imdbId, tmdbId)
links_map = {}
for _, row in links_df.iterrows():
    mid = int(row['movieId'])
    imdb = int(row['imdbId']) if not pd.isna(row['imdbId']) else None
    tmdb = int(row['tmdbId']) if not pd.isna(row['tmdbId']) else None
    links_map[mid] = (imdb, tmdb)

# 4. Insert Movies and populate Movie-Genre junction table
print("Inserting movies and parsing release years...")
cursor.execute("BEGIN TRANSACTION;")
for _, row in movies_df.iterrows():
    raw_mid = int(row['movieId'])
    raw_title = row['title']
    genres_str = row['genres']

    year, clean_title = extract_year(raw_title)
    imdb_id, tmdb_id = links_map.get(raw_mid, (None, None))

    cursor.execute("INSERT OR IGNORE INTO movie (movie_id, title, release_year, imdb_id, tmdb_id) VALUES (?, ?, ?, ?, ?);",
                   (raw_mid, clean_title, year, imdb_id, tmdb_id))

    if genres_str != "(no genres listed)":
        for g in genres_str.split('|'):
            if g in genre_map:
                cursor.execute("INSERT OR IGNORE INTO movie_genre (movie_id, genre_id) VALUES (?, ?);", (raw_mid, genre_map[g]))

cursor.execute("COMMIT;")
print("Movies and movie genre maps successfully inserted!")

conn.close()

Processing and inserting genres...
Inserted 19 unique genres.
Loading link metadata...
Inserting movies and parsing release years...
Movies and movie genre maps successfully inserted!


In [17]:
# 5. Insert Users and Ratings for Stage 1 (userId % 32 == 0) using Transactional Chunking with Datatype Casting
import os
import sqlite3
import pandas as pd
import gc

db_path = 'cinesearch.db'
data_dir = 'ml-32m-data/ml-32m'

# Forcefully garbage collect and close any lingering connection objects
# to completely release the SQLite database lock.
for obj in gc.get_objects():
    if isinstance(obj, sqlite3.Connection):
        try:
            obj.rollback()
            obj.close()
        except Exception:
            pass

# Redundant safety clean-up for the global namespace reference
if 'conn' in globals():
    try:
        conn.close()
    except Exception:
        pass

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ratings_path = os.path.join(data_dir, 'ratings.csv')
chunksize = 200_000
inserted_users = set()
inserted_ratings_count = 0

print("Beginning Transactional Bulk Load of Stage 1 ratings...")

try:
    for chunk in pd.read_csv(ratings_path, chunksize=chunksize):
        # Filter ratings deterministically for Stage 1
        subset = chunk[chunk['userId'] % 32 == 0]
        if subset.empty:
            continue

        # Identify unique users in this subset
        users_in_subset = set(subset['userId'].unique())
        new_users = users_in_subset - inserted_users

        cursor.execute("BEGIN TRANSACTION;")

        # Ingest new users (convert numpy.int64 to native Python int to avoid SQLite datatype mismatch)
        if new_users:
            cursor.executemany("INSERT OR IGNORE INTO app_user (user_id) VALUES (?);", [(int(uid),) for uid in new_users])
            inserted_users.update(new_users)

        # Bulk insert rating records (convert values to native Python types)
        ratings_tuples = [
            (int(row.userId), int(row.movieId), float(row.rating), int(row.timestamp))
            for row in subset.itertuples(index=False)
        ]
        cursor.executemany("INSERT OR IGNORE INTO rating (user_id, movie_id, rating, timestamp) VALUES (?, ?, ?, ?);", ratings_tuples)

        cursor.execute("COMMIT;")
        inserted_ratings_count += len(subset)
except Exception as e:
    try:
        cursor.execute("ROLLBACK;")
    except Exception:
        pass
    raise e
finally:
    conn.close()

print(f"\nTransactional Ingestion Completed!")
print(f"-> Ingested unique Users: {len(inserted_users):,}")
print(f"-> Ingested Stage 1 Ratings: {inserted_ratings_count:,}")

Beginning Transactional Bulk Load of Stage 1 ratings...

Transactional Ingestion Completed!
-> Ingested unique Users: 6,279
-> Ingested Stage 1 Ratings: 991,618


In [19]:
# 5. Insert Users and Ratings for Stage 1 (userId % 32 == 0) using Transactional Chunking with Datatype Casting
import os
import sqlite3
import pandas as pd
import gc

# Safely cleanup any dangling connections in the Python kernel to prevent database locks
for obj in gc.get_objects():
    if isinstance(obj, sqlite3.Connection):
        try:
            obj.rollback()
            obj.close()
        except Exception:
            pass

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ratings_path = os.path.join(data_dir, 'ratings.csv')
chunksize = 200_000
inserted_users = set()
inserted_ratings_count = 0

print("Beginning Transactional Bulk Load of Stage 1 ratings...")

try:
    for chunk in pd.read_csv(ratings_path, chunksize=chunksize):
        # Filter ratings deterministically for Stage 1
        subset = chunk[chunk['userId'] % 32 == 0]
        if subset.empty:
            continue

        # Identify unique users in this subset
        users_in_subset = set(subset['userId'].unique())
        new_users = users_in_subset - inserted_users

        cursor.execute("BEGIN TRANSACTION;")

        # Ingest new users (converting numpy.int64 to native Python int to avoid datatype mismatch)
        if new_users:
            cursor.executemany("INSERT OR IGNORE INTO app_user (user_id) VALUES (?);", [(int(uid),) for uid in new_users])
            inserted_users.update(new_users)

        # Bulk insert rating records (casting all NumPy types to standard Python int and float)
        ratings_tuples = [
            (int(row.userId), int(row.movieId), float(row.rating), int(row.timestamp))
            for row in subset.itertuples(index=False)
        ]
        cursor.executemany("INSERT OR IGNORE INTO rating (user_id, movie_id, rating, timestamp) VALUES (?, ?, ?, ?);", ratings_tuples)

        cursor.execute("COMMIT;")
        inserted_ratings_count += len(subset)
except Exception as e:
    try:
        cursor.execute("ROLLBACK;")
    except Exception:
        pass
    raise e
finally:
    conn.close()

print(f"\nTransactional Ingestion Completed!")
print(f"-> Ingested unique Users: {len(inserted_users):,}")
print(f"-> Ingested Stage 1 Ratings: {inserted_ratings_count:,}")

Beginning Transactional Bulk Load of Stage 1 ratings...

Transactional Ingestion Completed!
-> Ingested unique Users: 6,279
-> Ingested Stage 1 Ratings: 991,618


In [20]:
import sqlite3
import os

conn = sqlite3.connect('cinesearch.db')
cursor = conn.cursor()

print("=== Relational Schema Verification ===")
# 1. Inspect table counts
tables = ['movie', 'genre', 'movie_genre', 'app_user', 'rating']
for t in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {t};")
    cnt = cursor.fetchone()[0]
    print(f"Table '{t:<12}' | Row count: {cnt:,}")

# 2. Inspect Database File size
db_size_mb = os.path.getsize('cinesearch.db') / (1024 * 1024)
print(f"\nDatabase File Size on Disk: {db_size_mb:.2f} MB")

conn.close()

=== Relational Schema Verification ===
Table 'movie       ' | Row count: 87,585
Table 'genre       ' | Row count: 19
Table 'movie_genre ' | Row count: 147,090
Table 'app_user    ' | Row count: 6,279
Table 'rating      ' | Row count: 991,618

Database File Size on Disk: 46.61 MB


In [21]:
import os
import re
import sqlite3
import pandas as pd

db_path = 'cinesearch.db'
data_dir = 'ml-32m-data/ml-32m'

# Helper function to extract release year
def extract_year(title):
    match = re.search(r'\((\d{4})\)\s*$', title)
    if match:
        return int(match.group(1)), re.sub(r'\s*\(\d{4}\)\s*$', '', title).strip()
    return None, title.strip()

# 1. Establish database connection
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

print("Processing and inserting genres...")
# 2. Extract and insert unique genres
movies_df = pd.read_csv(os.path.join(data_dir, 'movies.csv'))
all_genres = set()
for genres_str in movies_df['genres'].dropna():
    if genres_str != "(no genres listed)":
        all_genres.update(genres_str.split('|'))

genre_map = {}
cursor.execute("BEGIN TRANSACTION;")
for g in sorted(all_genres):
    cursor.execute("INSERT OR IGNORE INTO genre (genre_name) VALUES (?);", (g,))
cursor.execute("COMMIT;")

# Build map of genre_name to genre_id
cursor.execute("SELECT genre_id, genre_name FROM genre;")
for gid, gname in cursor.fetchall():
    genre_map[gname] = gid

print(f"Inserted {len(genre_map)} unique genres.")

# 3. Load Links for O(1) metadata lookup
print("Loading link metadata...")
links_df = pd.read_csv(os.path.join(data_dir, 'links.csv'))
# Convert to dictionary mapping movieId -> (imdbId, tmdbId)
links_map = {}
for _, row in links_df.iterrows():
    mid = int(row['movieId'])
    imdb = int(row['imdbId']) if not pd.isna(row['imdbId']) else None
    tmdb = int(row['tmdbId']) if not pd.isna(row['tmdbId']) else None
    links_map[mid] = (imdb, tmdb)

# 4. Insert Movies and populate Movie-Genre junction table
print("Inserting movies and parsing release years...")
cursor.execute("BEGIN TRANSACTION;")
for _, row in movies_df.iterrows():
    raw_mid = int(row['movieId'])
    raw_title = row['title']
    genres_str = row['genres']

    year, clean_title = extract_year(raw_title)
    imdb_id, tmdb_id = links_map.get(raw_mid, (None, None))

    cursor.execute("INSERT OR IGNORE INTO movie (movie_id, title, release_year, imdb_id, tmdb_id) VALUES (?, ?, ?, ?, ?);",
                   (raw_mid, clean_title, year, imdb_id, tmdb_id))

    if genres_str != "(no genres listed)":
        for g in genres_str.split('|'):
            if g in genre_map:
                cursor.execute("INSERT OR IGNORE INTO movie_genre (movie_id, genre_id) VALUES (?, ?);", (raw_mid, genre_map[g]))

cursor.execute("COMMIT;")
print("Movies and movie genre maps successfully inserted!")

conn.close()

Processing and inserting genres...
Inserted 19 unique genres.
Loading link metadata...
Inserting movies and parsing release years...
Movies and movie genre maps successfully inserted!


In [24]:
# 5. Insert Users and Ratings for Stage 1 (userId % 32 == 0) using Transactional Chunking with Datatype Casting
import os
import sqlite3
import pandas as pd
import gc

# Safely cleanup any lingering connection objects in the kernel to release the database lock
for obj in gc.get_objects():
    if isinstance(obj, sqlite3.Connection):
        try:
            obj.rollback()
            obj.close()
        except Exception:
            pass

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ratings_path = os.path.join(data_dir, 'ratings.csv')
chunksize = 200_000
inserted_users = set()
inserted_ratings_count = 0

print("Beginning Transactional Bulk Load of Stage 1 ratings...")

try:
    for chunk in pd.read_csv(ratings_path, chunksize=chunksize):
        # Filter ratings deterministically for Stage 1
        subset = chunk[chunk['userId'] % 32 == 0]
        if subset.empty:
            continue

        # Identify unique users in this subset
        users_in_subset = set(subset['userId'].unique())
        new_users = users_in_subset - inserted_users

        cursor.execute("BEGIN TRANSACTION;")

        # Ingest new users (converting numpy.int64 to native Python int to avoid datatype mismatch)
        if new_users:
            cursor.executemany("INSERT OR IGNORE INTO app_user (user_id) VALUES (?);", [(int(uid),) for uid in new_users])
            inserted_users.update(new_users)

        # Bulk insert rating records (casting all NumPy types to standard Python int and float)
        ratings_tuples = [
            (int(row.userId), int(row.movieId), float(row.rating), int(row.timestamp))
            for row in subset.itertuples(index=False)
        ]
        cursor.executemany("INSERT OR IGNORE INTO rating (user_id, movie_id, rating, timestamp) VALUES (?, ?, ?, ?);", ratings_tuples)

        cursor.execute("COMMIT;")
        inserted_ratings_count += len(subset)
except Exception as e:
    try:
        cursor.execute("ROLLBACK;")
    except Exception:
        pass
    raise e
finally:
    conn.close()

print(f"\nTransactional Ingestion Completed!")
print(f"-> Ingested unique Users: {len(inserted_users):,}")
print(f"-> Ingested Stage 1 Ratings: {inserted_ratings_count:,}")

Beginning Transactional Bulk Load of Stage 1 ratings...

Transactional Ingestion Completed!
-> Ingested unique Users: 6,279
-> Ingested Stage 1 Ratings: 991,618


In [25]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("=== Relational Schema Verification ===")
# 1. Inspect table counts
tables = ['movie', 'genre', 'movie_genre', 'app_user', 'rating']
for t in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {t};")
    cnt = cursor.fetchone()[0]
    print(f"Table '{t:<12}' | Row count: {cnt:,}")

# 2. Inspect Database File size
db_size_mb = os.path.getsize(db_path) / (1024 * 1024)
print(f"\nDatabase File Size on Disk: {db_size_mb:.2f} MB")

conn.close()

=== Relational Schema Verification ===
Table 'movie       ' | Row count: 87,585
Table 'genre       ' | Row count: 19
Table 'movie_genre ' | Row count: 147,090
Table 'app_user    ' | Row count: 6,279
Table 'rating      ' | Row count: 991,618

Database File Size on Disk: 46.61 MB
